In [6]:
import os
import fastf1
import torch

# --- CONFIGURAZIONE PERCORSI (LOCALE SU PC) ---

# 1. Calcoliamo la root del progetto (saliamo di 3 livelli da dove siamo)
# Supponendo che il notebook sia in: F1/notebooks/sandboxes/gianluca
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, "../../.."))

# 2. Percorso Dati RAW (dove ci sono i file .parquet)
data_dir = os.path.join(project_root, "data", "raw")

# 3. Percorso Dati PROCESSATI (dove hai messo i file .pt scaricati)
processed_dir = os.path.join(project_root, "data", "processed_tensors")

# 4. Percorso CACHE (Creiamo una cartella cache dentro il progetto)
CACHE_DIR = os.path.join(project_root, "fastf1_cache")
if not os.path.exists(CACHE_DIR):
    os.makedirs(CACHE_DIR)
fastf1.Cache.enable_cache(CACHE_DIR)

# 5. Device (Usa la tua GPU se c'è, altrimenti CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"📂 Project Root: {project_root}")
print(f"📂 Data Dir:     {data_dir}")
print(f"📂 Processed:    {processed_dir}")
print(f"⚙️ Device:       {device}")

📂 Project Root: c:\Users\gianl\FastF1\FDS-F1-2025-Prediction
📂 Data Dir:     c:\Users\gianl\FastF1\FDS-F1-2025-Prediction\data\raw
📂 Processed:    c:\Users\gianl\FastF1\FDS-F1-2025-Prediction\data\processed_tensors
⚙️ Device:       cpu


In [7]:
import warnings
# Zittisce tutti i FutureWarning (quelli rossi che vedi)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import os
from google.colab import drive

# 1. Collega Google Drive
drive.mount('/content/drive')

# 2. Imposta il percorso
project_root = '/content/drive/MyDrive/F1'

os.chdir(project_root)
print(f"Siamo dentro: {os.getcwd()}")

In [ ]:
# 1. Correggiamo la versione della libreria problematica
!pip install websocket-client>=1.7

# 2. IMPORTANTE: Su Colab, dopo aver toccato queste librerie base,
# spesso bisogna riavviare il "motore" (Runtime).
# Se vedi strani errori dopo, vai nel menu in alto:
# "Runtime" -> "Restart session" (o Riavvia sessione)

In [ ]:
# Installa FastF1 e PyArrow (che serve per i file parquet)
!pip install fastf1

In [ ]:
!dir

Training della CNN:

Setup, Imprtazioni e Iperparametri

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import fastf1
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm  # Barra di caricamento figa



# --- 2. IPERPARAMETRI ---
HYPERPARAMS = {
    "batch_size": 16,       # Numero di giri analizzati insieme
    "learning_rate": 0.0001,# Velocità di apprendimento (bassa e precisa)
    "epochs": 30,           # Numero massimo di cicli
    "patience": 5           # Stop se non migliora per 5 volte (Early Stopping)
}

Split Dataset

In [8]:
# --- 3. DEFINIZIONE DATASET (TRAIN / VAL / TEST) ---
import os


# 3. VERIFICA IMMEDIATA (Debug)
if os.path.exists(data_dir):
    print(f"✅ Cartella dati trovata: {data_dir}")
    files = os.listdir(data_dir)
    print(f"📄 File trovati: {len(files)} (es. {files[:3]})")
else:
    print(f"❌ ERRORE: La cartella non esiste: {data_dir}")
    print("Controlla dove hai caricato la cartella 'data' su Google Drive!")

all_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.parquet')])

# Rimuoviamo file corrotti noti per pulizia
# blacklist = ['Monaco', 'Australian']
# all_files = [f for f in all_files if not any(b in f for b in blacklist)]

# LOGICA DI SPLIT PERSONALIZZATA
# 1. TEST SET: Tutti i file che contengono "Abu Dhabi" (qualsiasi anno)
test_files = [f for f in all_files if "Abu Dhabi" in f]

# 2. RIMANENTI: Tutto ciò che NON è Abu Dhabi
remaining_files = [f for f in all_files if f not in test_files]

# 3. VALIDATION SET: Ultime 4 gare cronologiche dai rimanenti
val_count = 4
val_files = remaining_files[-val_count:]

# 4. TRAINING SET: Tutto il resto
train_files = remaining_files[:-val_count]

print(f"--- DATASET SPLIT PLAN ---")
print(f"TRAIN: {len(train_files)} gare (Il modello studia su queste)")
print(f"VAL:   {len(val_files)} gare (Il modello fa pratica su queste: {val_files})")
print(f"TEST:  {len(test_files)} gare (Esame finale su: {test_files})")

if len(test_files) == 0:
    print("WARNING: Non ho trovato file Abu Dhabi! Il test set è vuoto.")

✅ Cartella dati trovata: c:\Users\gianl\FastF1\FDS-F1-2025-Prediction\data\raw
📄 File trovati: 32 (es. ['2023_Abu Dhabi Grand Prix_laps.parquet', '2023_Bahrain Grand Prix_laps.parquet', '2023_Hungarian Grand Prix_laps.parquet'])
--- DATASET SPLIT PLAN ---
TRAIN: 26 gare (Il modello studia su queste)
VAL:   4 gare (Il modello fa pratica su queste: ['2025_Mexico City Grand Prix_laps.parquet', '2025_Monaco Grand Prix_laps.parquet', '2025_Singapore Grand Prix_laps.parquet', '2025_Spanish Grand Prix_laps.parquet'])
TEST:  2 gare (Esame finale su: ['2023_Abu Dhabi Grand Prix_laps.parquet', '2024_Abu Dhabi Grand Prix_laps.parquet'])


Caricamento

In [ ]:
print("Caricamento dataset in memoria (CPU)...")
X_train, y_train = load_dataset_from_files(train_files, data_dir, desc="Training Set")
X_val, y_val = load_dataset_from_files(val_files, data_dir, desc="Validation Set")
X_test, y_test = load_dataset_from_files(test_files, data_dir, desc="Test Set (Abu Dhabi)")

print(f"\n📊 STATISTICHE DATASET:")
if X_train is not None: print(f"Train Samples: {len(X_train)}")
if X_val is not None: print(f"Val Samples:   {len(X_val)}")
if X_test is not None: print(f"Test Samples:  {len(X_test)}")

Salviamo il risultato sul drive così da non dover ogni volta aspettare 1.30h

In [ ]:
import torch
import os

# Definiamo una cartella su Drive per i dati pronti
processed_dir = '/content/drive/MyDrive/F1/data/processed_tensors'
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir)

print("💾 Salvataggio dei Tensor processati su Google Drive...")

# Salviamo Training Set
if X_train is not None:
    torch.save(X_train, os.path.join(processed_dir, 'X_train.pt'))
    torch.save(y_train, os.path.join(processed_dir, 'y_train.pt'))
    print("✅ Training Set salvato!")

# Salviamo Validation Set
if X_val is not None:
    torch.save(X_val, os.path.join(processed_dir, 'X_val.pt'))
    torch.save(y_val, os.path.join(processed_dir, 'y_val.pt'))
    print("✅ Validation Set salvato!")

# Salviamo Test Set
if X_test is not None:
    torch.save(X_test, os.path.join(processed_dir, 'X_test.pt'))
    torch.save(y_test, os.path.join(processed_dir, 'y_test.pt'))
    print("✅ Test Set salvato!")

print(f"\nTutto salvo in: {processed_dir}")
print("La prossima volta ci metterai 10 secondi a caricare questi file!")

Caricamento tensors dal Drive

In [9]:
try:
    print("Tentativo di caricamento rapido...")

    X_train = torch.load(os.path.join(processed_dir, 'X_train.pt'))
    y_train = torch.load(os.path.join(processed_dir, 'y_train.pt'))

    X_val = torch.load(os.path.join(processed_dir, 'X_val.pt'))
    y_val = torch.load(os.path.join(processed_dir, 'y_val.pt'))

    # Il Test set potrebbe non esserci se non hai trovato Abu Dhabi
    if os.path.exists(os.path.join(processed_dir, 'X_test.pt')):
        X_test = torch.load(os.path.join(processed_dir, 'X_test.pt'))
        y_test = torch.load(os.path.join(processed_dir, 'y_test.pt'))
    else:
        X_test, y_test = None, None

    print("✅ CARICAMENTO RAPIDO RIUSCITO! (Tempo risparmiato: 1 ora)")

except FileNotFoundError:
    print("⚠️ File processati non trovati. Avvio la procedura lenta (Download)...")
    # Qui chiami la funzione lenta vecchia
    X_train, y_train = load_dataset_from_files(train_files, data_dir, desc="Training Set")
    X_val, y_val = load_dataset_from_files(val_files, data_dir, desc="Validation Set")
    X_test, y_test = load_dataset_from_files(test_files, data_dir, desc="Test Set")

Tentativo di caricamento rapido...
✅ CARICAMENTO RAPIDO RIUSCITO! (Tempo risparmiato: 1 ora)


CHECK

In [5]:
import torch

print("🔍 CONTROLLO INTEGRITÀ DATI (SANITY CHECK)...")

# 1. Controlliamo se ci sono NaN (Not a Number) o Inf (Infiniti) negli Input
if torch.isnan(X_train).any():
    print("❌ ERRORE: Ci sono NaN in X_train! (Problema nel Preprocessing)")
elif torch.isinf(X_train).any():
    print("❌ ERRORE: Ci sono numeri Infiniti in X_train! (Divisione per zero?)")
else:
    print("✅ X_train è pulito (niente NaN/Inf).")

# 2. Controlliamo le Etichette (Devono essere solo 0 o 1)
print(f"Valori unici nelle etichette (y_train): {torch.unique(y_train)}")

if torch.isnan(y_train).any():
    print("❌ ERRORE: Ci sono NaN in y_train!")
else:
    print("✅ y_train è pulito.")

# 3. Controllo Forme
print(f"Shape X: {X_train.shape}")
print(f"Shape y: {y_train.shape}")

🔍 CONTROLLO INTEGRITÀ DATI (SANITY CHECK)...
✅ X_train è pulito (niente NaN/Inf).
Valori unici nelle etichette (y_train): tensor([0., 1.])
✅ y_train è pulito.
Shape X: torch.Size([9425, 4, 1000])
Shape y: torch.Size([9425, 1])


Inizializzazione Modello

In [ ]:
# --- 5. PREPARAZIONE TRAINING ---
from CNN import DriverStyleCNN

# Creazione DataLoader (Batching)
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=HYPERPARAMS['batch_size'], shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=HYPERPARAMS['batch_size'])
if X_test is not None:
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=HYPERPARAMS['batch_size'])

# Inizializzazione Rete (dal file src/models/cnn.py)
model = DriverStyleCNN().to(device)

# Loss e Optimizer
criterion = nn.BCELoss() # Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=HYPERPARAMS['learning_rate'])

# Variabili per tracciare la storia (per i grafici)
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
patience_counter = 0

Training

Grid Search

In [ ]:
import itertools
import torch
import json
import matplotlib.pyplot as plt
from preprocessing import train_one_model

# --- GRIGLIA ---
param_grid = {
    'batch_size': [16, 32],
    'learning_rate': [0.001, 0.0001],
    'patience': [5],
    'epochs': [30]
}

keys, values = zip(*param_grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"🚀 INIZIO GRID SEARCH: {len(combinations)} combinazioni.\n")

best_global_loss = float('inf')
best_global_config = None
best_global_history = None # Qui salveremo i dati per il grafico
best_global_model_path = "best_grid_model.pth"
best_global_config_path = "best_grid_config.json" # <--- NUOVO FILE

for i, config in enumerate(combinations):
    print(f"[{i+1}/{len(combinations)}] Test Config: {config} ...", end="")

    # Ora riceviamo anche 'history'
    loss, model_state, history = train_one_model(config, X_train, y_train, X_val, y_val, device)

    print(f" -> Val Loss: {loss:.4f} | Max Acc: {max(history['val_acc']):.2f}%")

    if loss < best_global_loss:
        best_global_loss = loss
        best_global_config = config
        best_global_history = history # Salviamo la storia del vincitore

        torch.save(model_state, best_global_model_path)
        with open(best_global_config_path, 'w') as f:
            json.dump(best_global_config, f)
            
        print("   🏆 RECORD! Modello e Config salvati.")

print("\n" + "="*40)
print(f"✅ VINCITORE: Loss {best_global_loss:.4f}")
print(f"Configurazione salvata in: {best_global_config_path}")
print("="*40)

🚀 INIZIO GRID SEARCH: 4 combinazioni.

[1/4] Test Config: {'batch_size': 16, 'learning_rate': 0.001, 'patience': 5, 'epochs': 30} ... -> Val Loss: 0.4057 | Max Acc: 82.97%
   🏆 RECORD! Modello e Config salvati.
[2/4] Test Config: {'batch_size': 16, 'learning_rate': 0.0001, 'patience': 5, 'epochs': 30} ... -> Val Loss: 0.9537 | Max Acc: 84.37%
[3/4] Test Config: {'batch_size': 32, 'learning_rate': 0.001, 'patience': 5, 'epochs': 30} ... -> Val Loss: 0.6610 | Max Acc: 83.04%
[4/4] Test Config: {'batch_size': 32, 'learning_rate': 0.0001, 'patience': 5, 'epochs': 30} ... -> Val Loss: 0.8872 | Max Acc: 84.02%

✅ VINCITORE: Loss 0.4057
Configurazione salvata in: best_grid_config.json


: 

Test finale su Abu Dhabi

In [11]:
from CNN import DriverStyleCNN
import os
import json
from torch.utils.data import TensorDataset, DataLoader

best_global_model_path = "best_grid_model.pth"
best_global_config_path = "best_grid_config.json"

# Controlliamo se abbiamo la configurazione in memoria, altrimenti la carichiamo dal file
if 'best_global_config' not in locals():
    print("⚠️ Nessuna configurazione in memoria. Cerco il file salvato...")
    if os.path.exists(best_global_config_path):
        with open(best_global_config_path, 'r') as f:
            best_global_config = json.load(f)
        print(f"✅ Configurazione caricata da file: {best_global_config}")
    else:
        raise FileNotFoundError("NON TROVO IL FILE DI CONFIGURAZIONE! Devi rifare il training.")

# Ora possiamo fare il test
if X_test is not None:
    print("\n🏁 PROVA DEL FUOCO SU ABU DHABI (Test Set)")
    
    # 1. Ricreiamo l'architettura (vuota)
    final_model = DriverStyleCNN().to(device)
    
    # 2. Carichiamo i pesi addestrati
    if os.path.exists(best_global_model_path):
        final_model.load_state_dict(torch.load(best_global_model_path, map_location=device))
        final_model.eval()
        print("✅ Modello caricato con successo.")
    else:
        raise FileNotFoundError("NON TROVO IL MODELLO .pth!")
    
    # 3. Creiamo il DataLoader usando il BATCH SIZE VINCENTE
    test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=best_global_config['batch_size'])
    
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = final_model(inputs)
            predicted = (outputs > 0.5).float()
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"ACCURATEZZA FINALE (Modello Ottimizzato): {100 * correct / total:.2f}%")
else:
    print("Nessun dato di test caricato.")


🏁 PROVA DEL FUOCO SU ABU DHABI (Test Set)
✅ Modello caricato con successo.
ACCURATEZZA FINALE (Modello Ottimizzato): 71.96%


MOMENTANEOOOO

In [ ]:
# PERCORSO ESATTO (preso dal tuo screenshot dell'errore)
filename = "/content/drive/MyDrive/F1/notebooks/sandboxes/gianluca/preprocessing.py"

content = """
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import fastf1
from tqdm.notebook import tqdm
import os
import gc
import sys
import copy
from scipy.interpolate import interp1d

from notebooks.sandboxes.gianluca.CNN import DriverStyleCNN

# --- COSTANTI DI NORMALIZZAZIONE ---
PHYSICAL_LIMITS = {
    'Speed': 365.0,
    'RPM': 13500.0,
    'Throttle': 100.0,
    'Brake': 100.0,
    'nGear': 8.0
}

# --- 1. FUNZIONI DI PREPROCESSING ---
def process_telemetry(lap, target_length=1000):
    try:
        tel = lap.get_telemetry()
        required_cols = ['Distance', 'Speed', 'RPM', 'Throttle', 'Brake']

        if not all(col in tel.columns for col in required_cols): return None
        if len(tel) < 50: return None

        # Interpolazione NaN
        tel = tel[required_cols].interpolate(method='linear', limit_direction='both')
        tel = tel.ffill().bfill().fillna(0)

        # Resampling
        total_dist = tel['Distance'].max()
        x_original = tel['Distance'].values
        x_new = np.linspace(0, total_dist, target_length)

        processed_channels = []
        keys = ['Speed', 'RPM', 'Throttle', 'Brake']

        for key in keys:
            values = tel[key].values
            f = interp1d(x_original, values, kind='linear', fill_value="extrapolate")
            new_values = f(x_new)

            if key == 'Brake':
                max_val = np.max(new_values)
                norm_values = np.clip(new_values / 100.0, 0.0, 1.0) if max_val > 1.5 else np.clip(new_values, 0.0, 1.0)
            else:
                limit = PHYSICAL_LIMITS[key]
                norm_values = np.clip(new_values / limit, 0.0, 1.0)

            processed_channels.append(norm_values)

        return np.array(processed_channels, dtype=np.float32)
    except Exception:
        return None

# --- 2. FUNZIONE CARICAMENTO DATI ---
def load_dataset_from_files(file_list, data_dir, desc="Loading"):
    X_list = []
    y_list = []

    import warnings
    warnings.simplefilter(action='ignore', category=FutureWarning)

    if not os.path.exists(data_dir): return None, None

    pbar = tqdm(file_list, desc=desc)

    for file_name in pbar:
        try:
            path = os.path.join(data_dir, file_name)
            df = pd.read_parquet(path)
            if df.empty: continue

            parts = file_name.split('_')
            year, race_name = int(parts[0]), parts[1]

            for session_name, group in df.groupby('Session'):
                session = None
                try:
                    session = fastf1.get_session(year, race_name, session_name)
                    session.load(telemetry=True, weather=False, messages=False)

                    for _, row in group.iterrows():
                        try:
                            drv_laps = session.laps[session.laps['Driver'] == row['Driver']]
                            lap_data = drv_laps[drv_laps['LapNumber'] == row['LapNumber']]
                            if lap_data.empty: continue

                            matrix = process_telemetry(lap_data.iloc[0])
                            if matrix is not None:
                                X_list.append(matrix)
                                y_list.append(row['Label'])
                        except: continue
                except: continue
                finally:
                    if session: del session
                    gc.collect()
            del df
            gc.collect()
        except: continue

    if len(X_list) == 0: return None, None

    X_np = np.array(X_list, dtype=np.float32)
    y_np = np.array(y_list, dtype=np.float32)

    del X_list, y_list
    gc.collect()

    return torch.from_numpy(X_np), torch.from_numpy(y_np).unsqueeze(1)

# --- 3. FUNZIONE DI TRAINING ---
def train_one_model(config, X_train, y_train, X_val, y_val, device):

    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=config['batch_size'])

    # Tentativo di inizializzare il modello (gestisce nomi diversi)
    from notebooks.sandboxes.gianluca.CNN import DriverStyleCNN

    model = DriverStyleCNN().to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])

    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    # History per i grafici
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(config['epochs']):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                val_loss += criterion(outputs, labels).item()
                predicted = (outputs > 0.5).float()
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        val_acc = 100 * correct / total

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)

        if avg_val_loss < best_loss:
            best_loss = avg_val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config['patience']:
                break

    return best_loss, best_model_state, history
"""

with open(filename, "w") as f:
    f.write(content)

print("✅ FILE PREPROCESSING AGGIORNATO!")